[Guide Index](README.md)


# 06b Working with Backend Graph Databases

Most examples in the StarLayer user guide demonstrate the functionality of `StarLayerGraph` using the in-memory graph. `StarLayerGraph` is the working name of the Oxigraph-backed graph class that becomes `StarLayerGraph` at the switch (see [the Oxigraph-backed graph guide](02d-oxigraph-wrapper.ipynb)); it wraps a `pyoxigraph.Store`, so the graph is always held by Oxigraph in the same process. There are two ways to persist data, and a third thing that is no longer there:
- **An on-disk Oxigraph store**: `StarLayerGraph(path=...)` / `StarLayerDataset(path=...)` open a RocksDB store in a directory.
- **A SPARQL HTTP endpoint** (Oxigraph server, Fuseki, ...): talked to with the HTTP functions in `starlayer.graph.backends.native`, not through a graph object.
- The old class's other stores (`SQLAlchemy`, any rdflib `Store` plugin, `SPARQLUpdateStore`) and its `backend="rdf-1.1" / "rdf-1.2"` argument are dropped: an `StarLayerGraph` does not take a `store=` of that kind or a `backend=`. The old distinction between RDF 1.2 and RDF 1.1 endpoints is handled where it matters, at the moment data is exported (section 1.1).

The examples below show both.

## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01-getting-started.ipynb). Run cells from top to bottom; later cells reuse variables from earlier ones.

Section 1 needs a running Oxigraph and/or Fuseki instance (each example skips cleanly if its endpoint isn't reachable) — every example clears all existing data in the dataset it connects to (`CLEAR ALL` / equivalent), so don't point one at a database you care about. Section 2 needs nothing but a writable directory.

The outputs saved in this notebook were produced without a server running, so the section 1 cells show the "skipping" message; against a live endpoint they print the graph read back.

In [1]:
import requests
from rdflib import Namespace
from starlayer.graph import TripleTerm
from starlayer.graph.oxgraph import StarLayerGraph
from starlayer.graph.backends import native

EX = Namespace("http://example.org/")

## 1. Using a SPARQL endpoint

An `StarLayerGraph` does not connect to a remote store. To use a SPARQL server, build the data in an `StarLayerGraph`, send it with a SPARQL update, and read results back into an `StarLayerGraph`. `starlayer.graph.backends.native` has the HTTP functions for this: `http_update(update_url, sparql, headers)`, `http_select`, `http_ask` and `http_construct`. They post the SPARQL text unchanged, so SPARQL 1.2 syntax (triple terms, `TRIPLE()`, base-direction functions) is passed straight through. `http_select` returns `(vars, bindings)` with rdflib terms; `http_construct` returns the endpoint's Turtle, which `StarLayerGraph.parse()` reads. Credentials, if any, go in the `headers` argument (here an `Authorization` header).

StarLayer has been tested against both Oxigraph and Fuseki. Both support RDF 1.2 and RDF 1.1 modes. RDF 1.2 is a better choice when working with RDF 1.2 graphs; RDF 1.1 is for integrating RDF 1.2 data with an existing SPARQL service that operates using RDF 1.1 (section 1.1).

The following examples require a running instance of Oxigraph and/or Fuseki.  Initiating a session is outside the scope of this user guide.

The examples assume the following end-points.  Your end-points may differ.
- Oxigraph: `http://localhost:7878/`
- Fuseki: `http://localhost:3030/starlayergraph` (5.5+ needed for native RDF 1.2)


In [2]:
import base64

def basic_auth(user, password):
    token = base64.b64encode(f"{user}:{password}".encode()).decode()
    return {"Authorization": f"Basic {token}"}

def demo_sparql_endpoint(label, query_endpoint, update_endpoint, headers=None, rdf_version=None):
    """Connect, clear, write a reification, read it back into an StarLayerGraph and print it."""
    g = StarLayerGraph(identifier=EX.main)
    g.bind("ex", EX)
    g.add_reification(EX.claim, TripleTerm(EX.bob, EX.knows, EX.carol))
    # N-Triples text is valid inside SPARQL INSERT DATA. rdf_version="1.1" exports triple terms in the
    # basic encoding, for an endpoint that does not know RDF 1.2.
    triples = g.serialize(format="nt", rdf_version=rdf_version)
    try:
        native.http_update(update_endpoint, "CLEAR ALL", headers)
    except requests.exceptions.ConnectionError:
        print(f"{label}: skipping - no live instance at {query_endpoint}")
        return
    native.http_update(update_endpoint, f"INSERT DATA {{ GRAPH <{EX.main}> {{ {triples} }} }}", headers)

    body, _content_type = native.http_construct(
        query_endpoint, f"CONSTRUCT {{ ?s ?p ?o }} WHERE {{ GRAPH <{EX.main}> {{ ?s ?p ?o }} }}", headers)
    back = StarLayerGraph(identifier=EX.main)
    back.bind("ex", EX)
    back.parse(data=body.decode("utf-8"), format="turtle")
    print(f"{label}:")
    print(back.serialize(format="turtle", pretty=True))

# test oxigraph
demo_sparql_endpoint(
    "Oxigraph",
    query_endpoint="http://localhost:7878/query",
    update_endpoint="http://localhost:7878/update",
)

# test fuseki
demo_sparql_endpoint(
    "Fuseki",
    query_endpoint="http://localhost:3030/starlayergraph/query",
    update_endpoint="http://localhost:3030/starlayergraph/update",
    headers=basic_auth("admin", "admin"),
)

Oxigraph: skipping - no live instance at http://localhost:7878/query
Fuseki: skipping - no live instance at http://localhost:3030/starlayergraph/query


### 1.1 RDF 1.1 SPARQL endpoint

The old class had a `backend="rdf-1.1"` mode. It is gone; an endpoint that only knows RDF 1.1 is served by exporting the data in RDF 1.1 form: `serialize(rdf_version="1.1")` writes each triple term in the "basic encoding" (an `rdf:PropositionForm` node with `rdf:propositionFormSubject`, `rdf:propositionFormPredicate` and `rdf:propositionFormObject`), so the text contains no `<<( )>>`. The cell below prints that export, then runs the same round trip against an endpoint. What the endpoint stores is the encoded form, so what comes back is the encoded triples, not a triple term.

In [3]:
demo = StarLayerGraph()
demo.bind("ex", EX)
demo.add_reification(EX.claim, TripleTerm(EX.bob, EX.knows, EX.carol))
print(demo.serialize(format="turtle", rdf_version="1.1"))

demo_sparql_endpoint(
    "Oxigraph, RDF 1.1 export",
    query_endpoint="http://localhost:7878/query",
    update_endpoint="http://localhost:7878/update",
    rdf_version="1.1",
)

@prefix ex: <http://example.org/> .
ex:claim <http://www.w3.org/1999/02/22-rdf-syntax-ns#reifies> _:eac224576a0d96d310b6496f5d85090b .
_:eac224576a0d96d310b6496f5d85090b a <http://www.w3.org/1999/02/22-rdf-syntax-ns#PropositionForm> ;
	<http://www.w3.org/1999/02/22-rdf-syntax-ns#propositionFormSubject> ex:bob ;
	<http://www.w3.org/1999/02/22-rdf-syntax-ns#propositionFormPredicate> ex:knows ;
	<http://www.w3.org/1999/02/22-rdf-syntax-ns#propositionFormObject> ex:carol .

Oxigraph, RDF 1.1 export: skipping - no live instance at http://localhost:7878/query


### 1.2 Reasoning-enabled Fuseki

Entailment support is entirely a property of how the *endpoint itself* is configured, so nothing about the HTTP call changes: send the query with `http_select` and the endpoint answers with its reasoning applied. (The old class had a `query(..., entailment="native")` label for this; `entailment=` is not available on the Oxigraph wrapper yet, and the label never changed how the query was sent.) See [fuseki-reasoning-setup.md](../../packages/starlayer/docs/graph/fuseki-reasoning-setup.md) for how to stand up a Fuseki instance that reasons before answering - this example expects one at `localhost:3031`, a dataset named `starlayergraph-rdfs`, separate from the plain one above, started with `--rdfs=schema.ttl` declaring `ex:Manager rdfs:subClassOf ex:Employee`. See the [Graph Inferencing guide](02b-graphs-inferencing.ipynb) for `infer()` - StarLayer's own reasoning on an `StarLayerGraph`, which needs none of this setup.

In [4]:
query_url = "http://localhost:3031/starlayergraph-rdfs/query"
update_url = "http://localhost:3031/starlayergraph-rdfs/update"
try:
    native.http_update(update_url, "CLEAR ALL")
except requests.exceptions.ConnectionError:
    print("skipping - no live reasoning-enabled instance at localhost:3031")
else:
    native.http_update(update_url, "PREFIX ex: <http://example.org/> INSERT DATA { ex:alice a ex:Manager }")

    _vars, rows = native.http_select(
        query_url,
        "PREFIX ex: <http://example.org/> SELECT ?x WHERE { ?x a ex:Employee }",
    )
    print("alice is an Employee (via rdfs:subClassOf, reasoned by Fuseki itself):", [str(r["x"]) for r in rows])

skipping - no live reasoning-enabled instance at localhost:3031


## 2. Using an on-disk store

`StarLayerGraph(path=...)` opens (or creates) an Oxigraph store in a directory, so the data survives the process. Only one process may write to a path at a time: drop the first object before opening the path again. The wrapper has no `open()` / `commit()` / `close()`; writes are applied as they are made (`g.store.flush()` forces them to disk, see [the Oxigraph-backed graph guide](02d-oxigraph-wrapper.ipynb)). The old `SQLAlchemy` store and other rdflib store plugins are dropped.

In [5]:
import tempfile

db_path = tempfile.mkdtemp()

writer = StarLayerGraph(path=db_path)
writer.bind("ex", EX)
writer.add_reification(EX.claim, TripleTerm(EX.bob, EX.knows, EX.carol))
writer.store.flush()
del writer  # release the single-writer lock on the directory

# fresh graph object, same directory - proves the data actually persisted
reader = StarLayerGraph(path=db_path)
reader.bind("ex", EX)
print(len(reader), "triple read back from disk")
print(reader.serialize(format="turtle", pretty=True))

1 triple read back from disk
@version "1.2" .
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

ex:claim rdf:reifies <<( ex:bob ex:knows ex:carol )>> .

